# Exploratory Data Analysis

This notebook avoids profiling and avoids building the large joined tables from the original notebook. It focuses on targeted, scalable exploratory analysis for movie recommendation insights.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## Load CSV Files

In [ ]:
csv_folder = Path("./movies-database")

movies       = pd.read_csv(csv_folder / "movies.csv")
links        = pd.read_csv(csv_folder / "links.csv")
ratings      = pd.read_csv(csv_folder / "ratings.csv")
tags         = pd.read_csv(csv_folder / "tags.csv")
genome_scores = pd.read_csv(csv_folder / "genome-scores.csv")
genome_tags  = pd.read_csv(csv_folder / "genome-tags.csv")

## Explorative analysis

In [ ]:
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)$")[0].astype("float")
ratings["rating_year"] = pd.to_datetime(ratings["timestamp"], unit="s").dt.year
tags["tag_clean"] = tags["tag"].astype("string").str.lower().str.strip()

### 1. Movie-level rating summary
Aggregate ratings once at movie level. Most later analyses use this smaller table instead of reprocessing all raw ratings.

In [ ]:
movie_stats = (
    ratings.groupby("movieId")
    .agg(
        avg_rating=("rating", "mean"),
        num_ratings=("rating", "size"),
        rating_std=("rating", "std")
    )
    .reset_index()
    .merge(movies[["movieId", "title", "genres", "year"]], on="movieId", how="left")
)

movie_stats.head()

### 2. Highest-rated genres
Explode genres only from the compact movie-level summary, not from the full ratings table.

In [ ]:
movie_genres = movie_stats.assign(genre=movie_stats["genres"].str.split("|")).explode("genre")
movie_genres = movie_genres[movie_genres["genre"] != "(no genres listed)"]

genre_summary = (
    movie_genres.groupby("genre")
    .agg(
        avg_movie_rating=("avg_rating", "mean"),
        median_movie_rating=("avg_rating", "median"),
        avg_num_ratings=("num_ratings", "mean"),
        movie_count=("movieId", "nunique")
    )
    .reset_index()
    .sort_values("avg_movie_rating", ascending=False)
)

genre_summary

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=genre_summary, x="avg_movie_rating", y="genre", color="steelblue")
plt.title("Average Movie Rating by Genre")
plt.xlabel("Average movie rating")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

### 3. Most popular genres
This highlights engagement: which genres tend to attract the most ratings per movie.

In [ ]:
genre_popularity = (
    movie_genres.groupby("genre")
    .agg(
        total_movies=("movieId", "nunique"),
        avg_num_ratings=("num_ratings", "mean"),
        median_num_ratings=("num_ratings", "median")
    )
    .reset_index()
    .sort_values("avg_num_ratings", ascending=False)
)

genre_popularity

### 4. Best popular movies
Restrict to movies with many ratings so that the list reflects broad audience approval rather than tiny samples.

In [ ]:
best_movies = (
    movie_stats[movie_stats["num_ratings"] >= 1000]
    .sort_values(["avg_rating", "num_ratings"], ascending=[False, False])
    [["title", "year", "genres", "avg_rating", "num_ratings"]]
)

best_movies.head(20)

### 5. Most divisive popular movies
High standard deviation suggests polarizing audience reactions.

In [ ]:
divisive_movies = (
    movie_stats[movie_stats["num_ratings"] >= 1000]
    .sort_values("rating_std", ascending=False)
    [["title", "genres", "avg_rating", "rating_std", "num_ratings"]]
)

divisive_movies.head(20)

### 6. Ratings activity over time
Shows how user engagement and average rating behavior changed over the years when ratings were submitted.

In [ ]:
ratings_by_year = (
    ratings.groupby("rating_year")
    .agg(
        total_ratings=("rating", "size"),
        avg_rating=("rating", "mean"),
        unique_users=("userId", "nunique"),
        unique_movies=("movieId", "nunique")
    )
    .reset_index()
    .sort_values("rating_year")
)

ratings_by_year

In [ ]:
plt.figure(figsize=(11, 5))
sns.lineplot(data=ratings_by_year, x="rating_year", y="total_ratings")
plt.title("Ratings Activity Over Time")
plt.xlabel("Year of rating submission")
plt.ylabel("Number of ratings")
plt.tight_layout()
plt.show()

### 7. Most-tagged movies
Tags provide useful qualitative signals about what viewers notice and talk about.

In [ ]:
tagged_movies = (
    tags.groupby("movieId")
    .agg(
        num_tags=("tag", "size"),
        unique_taggers=("userId", "nunique"),
        unique_tags=("tag_clean", "nunique")
    )
    .reset_index()
    .merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")
    .sort_values("num_tags", ascending=False)
)

tagged_movies.head(20)

### 8. Most common tags
These are recurring audience descriptors that may later become useful recommendation features.

In [ ]:
common_tags = (
    tags.groupby("tag_clean")
    .agg(
        count=("tag_clean", "size"),
        unique_movies=("movieId", "nunique")
    )
    .reset_index()
    .sort_values("count", ascending=False)
)

common_tags.head(30)

### 9. Common tags associated with higher ratings
Restrict to common tags to keep the analysis more robust and more computationally manageable.

In [ ]:
popular_tag_names = common_tags.loc[common_tags["count"] >= 100, "tag_clean"]
tags_small = tags[tags["tag_clean"].isin(popular_tag_names)].copy()

tag_rating = (
    tags_small.merge(ratings[["userId", "movieId", "rating"]], on=["userId", "movieId"], how="inner")
    .groupby("tag_clean")
    .agg(
        avg_rating=("rating", "mean"),
        num_tagged_ratings=("rating", "size"),
        unique_movies=("movieId", "nunique")
    )
    .reset_index()
    .sort_values("avg_rating", ascending=False)
)

tag_rating.head(30)

### 10. Ratings by release decade
This shows whether older or newer movie eras tend to receive stronger average ratings.

In [ ]:
decade_summary = (
    movie_stats.dropna(subset=["year"])
    .assign(decade=lambda df: (df["year"] // 10 * 10).astype(int))
    .groupby("decade")
    .agg(
        avg_movie_rating=("avg_rating", "mean"),
        avg_num_ratings=("num_ratings", "mean"),
        movie_count=("movieId", "nunique")
    )
    .reset_index()
    .sort_values("decade")
)

decade_summary

In [ ]:
plt.figure(figsize=(11, 5))
sns.barplot(data=decade_summary, x="decade", y="avg_movie_rating", color="darkorange")
plt.title("Average Movie Rating by Release Decade")
plt.xlabel("Release decade")
plt.ylabel("Average movie rating")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()